# DuckDB — Practical Tour

DuckDB is an **in-process OLAP database**. Think SQLite but built for analytics:
columnar storage, vectorised execution, and zero-config setup.

In this sandbox it fills the gap between Spark (distributed) and pandas (in-memory):

```
pandas        → great for row manipulation, small data
DuckDB        → fast SQL analytics on files and DataFrames, no cluster needed
Spark         → distributed processing, large data that doesn't fit locally
```

### Sections
1. In-memory — query Python objects directly
2. Reading files — CSV, JSON, Parquet with glob patterns
3. DuckDB SQL extras — features you don't get in most databases
4. PostgreSQL — attach and query Postgres from DuckDB
5. Persistent file — save a `.duckdb` store across sessions
6. DuckDB ↔ pandas — round-trip patterns

In [1]:
import duckdb
import pandas as pd
import json, os, pathlib
from faker import Faker
import random

# Shared data directory (mounted from the host)
DATA = pathlib.Path('/home/jupyter/data')
(DATA / 'duck').mkdir(parents=True, exist_ok=True)

print(f'DuckDB version: {duckdb.__version__}')

DuckDB version: 1.5.4


## Sample data

Generate data once — all sections below reuse it.

In [2]:
fake = Faker()
Faker.seed(42)
random.seed(42)

CATS    = ['electronics', 'food', 'clothing', 'travel', 'health']
STATUS  = ['completed', 'pending', 'refunded']
COUNTRY = ['US', 'UK', 'MX', 'SG', 'DE', 'BR', 'FR']

users_df = pd.DataFrame([
    {'user_id': i, 'name': fake.name(), 'country': random.choice(COUNTRY),
     'email': fake.email(), 'signup_date': fake.date_between('-2y', 'today')}
    for i in range(1, 101)
])

txns_df = pd.DataFrame([
    {'txn_id': i, 'user_id': random.randint(1, 100),
     'amount': round(random.uniform(5, 800), 2),
     'category': random.choice(CATS), 'status': random.choice(STATUS),
     'created_at': fake.date_time_between('-90d', 'now')}
    for i in range(1, 2001)
])

print(f'Users:        {len(users_df):,}')
print(f'Transactions: {len(txns_df):,}')

Users:        100
Transactions: 2,000


---
## 1 · In-memory — query Python objects directly

DuckDB can query a **pandas DataFrame in place** — no copy, no loading.  
Just reference the variable name inside the SQL string.

In [3]:
# Query a DataFrame as if it were a table — variable name IS the table name
duckdb.sql("""
    SELECT country, COUNT(*) AS users
    FROM users_df
    GROUP BY ALL
    ORDER BY users DESC
""").show()

┌─────────┬───────┐
│ country │ users │
│ varchar │ int64 │
├─────────┼───────┤
│ US      │    19 │
│ BR      │    16 │
│ MX      │    16 │
│ UK      │    15 │
│ FR      │    13 │
│ DE      │    12 │
│ SG      │     9 │
└─────────┴───────┘



In [4]:
# Join two DataFrames with SQL — no merge(), no index juggling
duckdb.sql("""
    SELECT u.country,
           t.category,
           ROUND(SUM(t.amount), 2)  AS revenue,
           COUNT(*)                 AS tx_count
    FROM txns_df  t
    JOIN users_df u USING (user_id)
    WHERE t.status = 'completed'
    GROUP BY ALL
    ORDER BY revenue DESC
    LIMIT 10
""").df()   # .df() → pandas  |  .pl() → polars  |  .show() → print

,country,category,revenue,tx_count
0,US,health,16688.54,37
1,BR,health,12295.58,27
2,US,food,12009.80,33
3,MX,clothing,11206.60,27
4,MX,food,9586.40,22
5,UK,food,9510.96,22
6,US,electronics,9016.95,26
7,UK,travel,8898.65,24
8,FR,electronics,8801.05,20
9,BR,food,8702.64,21


In [5]:
# Use a named connection to register views explicitly
con = duckdb.connect()
con.register('users',   users_df)
con.register('txns',    txns_df)

# Now the names are fixed regardless of the Python variable name
con.sql('SELECT COUNT(*) AS n FROM txns').show()

┌───────┐
│   n   │
│ int64 │
├───────┤
│  2000 │
└───────┘



---
## 2 · Reading files — CSV, JSON, Parquet

DuckDB reads files **lazily** — it only scans what the query needs.  
Glob patterns let you query an entire directory as one table.

In [6]:
# Write sample files to disk so we can demo file queries
users_df.to_csv(DATA / 'duck/users.csv',  index=False)
txns_df.to_parquet(DATA / 'duck/txns.parquet', index=False)

# Write JSON lines for the JSON demo
with open(DATA / 'duck/events.jsonl', 'w') as f:
    for row in txns_df.head(200).to_dict(orient='records'):
        f.write(json.dumps(row, default=str) + '\n')

print('Files written.')

Files written.


In [7]:
# read_csv — DuckDB auto-detects types and header
duckdb.sql("SELECT * FROM read_csv('/home/jupyter/data/duck/users.csv') LIMIT 5").df()

,user_id,name,country,email,signup_date
0,1,Allison Hill,BR,donaldgarcia@example.net,2026-03-30
1,2,Javier Johnson,US,jesseguzman@example.net,2024-11-08
2,3,Kimberly Robinson,US,lisa02@example.net,2025-04-20
3,4,Daniel Gallagher,BR,daviscolin@example.com,2025-08-31
4,5,Monica Herrera,MX,smiller@example.net,2025-09-11


In [8]:
# read_parquet — glob pattern queries multiple files at once
# e.g. after a Spark job writes partitioned output:
#   read_parquet('/home/jupyter/data/output/transactions/**/*.parquet')
duckdb.sql("""
    SELECT category,
           ROUND(AVG(amount), 2) AS avg_amount,
           COUNT(*)              AS n
    FROM read_parquet('/home/jupyter/data/duck/txns.parquet')
    GROUP BY ALL
    ORDER BY avg_amount DESC
""").df()

,category,avg_amount,n
0,food,428.00,400
1,travel,406.46,425
2,health,405.19,432
3,clothing,398.51,372
4,electronics,392.80,371


In [9]:
# read_json — handles JSON lines automatically
duckdb.sql("""
    SELECT status, COUNT(*) AS n, ROUND(SUM(amount), 2) AS total
    FROM read_json('/home/jupyter/data/duck/events.jsonl')
    GROUP BY ALL
""").show()

┌───────────┬───────┬──────────┐
│  status   │   n   │  total   │
│  varchar  │ int64 │  double  │
├───────────┼───────┼──────────┤
│ refunded  │    65 │ 27045.07 │
│ completed │    72 │ 28808.61 │
│ pending   │    63 │ 27256.42 │
└───────────┴───────┴──────────┘



In [10]:
# Create a persistent view over a file — reuse it in the session
con.sql("CREATE OR REPLACE VIEW txns_file AS SELECT * FROM read_parquet('/home/jupyter/data/duck/txns.parquet')")
con.sql("SELECT COUNT(*) AS rows FROM txns_file").show()

┌───────┐
│ rows  │
│ int64 │
├───────┤
│  2000 │
└───────┘



---
## 3 · DuckDB SQL extras

DuckDB extends standard SQL with several quality-of-life features.

In [11]:
# SUMMARIZE — instant descriptive statistics for every column
con.sql("SUMMARIZE txns").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,txn_id,BIGINT,1,2000,2308,1000.5,577.4945887192364,500,1000,1500,2000,0.0
1,user_id,BIGINT,1,100,96,50.6995,29.25807952273491,25,51,76,2000,0.0
2,amount,DOUBLE,5.16,799.98,2149,406.4811449999996,225.39435671630886,211.64843387109343,405.13653136200713,600.6144784313725,2000,0.0
3,category,VARCHAR,clothing,travel,5,None,None,None,None,None,2000,0.0
4,status,VARCHAR,completed,refunded,3,None,None,None,None,None,2000,0.0
5,created_at,TIMESTAMP_NS,2026-03-19 22:33:49.043438,2026-06-17 20:53:11.920731,2184,2026-05-04 06:36:13.095608,None,2026-04-12 21:41:24.627101,2026-05-04 15:12:12.654636,2026-05-26 02:47:56.942693,2000,0.0


In [12]:
# GROUP BY ALL — group by every non-aggregated column automatically
con.sql("""
    SELECT country, category, status,
           ROUND(SUM(amount), 2) AS revenue
    FROM txns
    JOIN users USING (user_id)
    GROUP BY ALL
    ORDER BY revenue DESC
    LIMIT 8
""").df()

,country,category,status,revenue
0,US,health,completed,16688.54
1,US,food,refunded,16076.54
2,BR,health,refunded,13394.21
3,BR,travel,refunded,13238.98
4,US,health,pending,12860.57
5,BR,health,completed,12295.58
6,US,clothing,pending,12257.10
7,BR,travel,pending,12256.55


In [13]:
# SELECT * EXCLUDE   — return everything except named columns
# SELECT * REPLACE   — return everything but override specific columns
con.sql("""
    SELECT
        * EXCLUDE (email),                            -- drop email
        * REPLACE (ROUND(amount, 0)::INT AS amount)   -- round amount in place
    FROM txns
    JOIN users USING (user_id)
    LIMIT 5
""").df()

,txn_id,user_id,amount,category,status,created_at,name,country,signup_date,txn_id_1,user_id_1,amount_1,category_1,status_1,created_at_1,name_1,country_1,email,signup_date_1
0,1,88,520.19,health,refunded,2026-04-10 19:33:44.525640,David Mann,SG,2026-01-12,1,88,520,health,refunded,2026-04-10 19:33:44.525640,David Mann,SG,ojones@example.com,2026-01-12
1,2,22,429.64,food,completed,2026-05-08 11:50:10.237823,Michael Lewis,DE,2025-05-27,2,22,430,food,completed,2026-05-08 11:50:10.237823,Michael Lewis,DE,jenniferkhan@example.net,2025-05-27
2,3,60,306.67,health,completed,2026-04-23 09:38:57.839010,Gary Owens,US,2026-05-08,3,60,307,health,completed,2026-04-23 09:38:57.839010,Gary Owens,US,clarkrobert@example.net,2026-05-08
3,4,88,262.80,electronics,completed,2026-06-10 20:20:19.733868,David Mann,SG,2026-01-12,4,88,263,electronics,completed,2026-06-10 20:20:19.733868,David Mann,SG,ojones@example.com,2026-01-12
4,5,5,645.01,travel,pending,2026-05-04 14:54:14.173026,Monica Herrera,MX,2025-09-11,5,5,645,travel,pending,2026-05-04 14:54:14.173026,Monica Herrera,MX,smiller@example.net,2025-09-11


In [14]:
# QUALIFY — filter on window function results (no subquery needed)
# "top transaction per user per category"
con.sql("""
    SELECT user_id, category, amount, created_at
    FROM txns
    WHERE status = 'completed'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id, category ORDER BY amount DESC) = 1
    ORDER BY user_id, category
    LIMIT 10
""").df()

,user_id,category,amount,created_at
0,1,clothing,499.94,2026-04-04 13:55:21.338440
1,1,electronics,746.26,2026-05-03 10:04:30.326612
2,1,food,243.77,2026-05-14 16:43:07.452914
3,1,health,612.57,2026-06-01 01:31:23.549827
4,1,travel,488.35,2026-05-09 06:22:38.454561
5,2,electronics,590.86,2026-03-22 22:09:46.535713
6,2,food,585.36,2026-04-15 11:28:27.990458
7,2,health,594.44,2026-04-26 03:49:47.294903
8,3,electronics,258.38,2026-03-25 11:37:52.280048
9,3,food,543.01,2026-04-12 00:47:44.708665


In [15]:
# PIVOT — reshape rows to columns without a CASE WHEN cascade
con.sql("""
    PIVOT (
        SELECT category, status, ROUND(SUM(amount), 2) AS total
        FROM txns
        GROUP BY ALL
    )
    ON status
    USING SUM(total)
    GROUP BY category
    ORDER BY category
""").df()

,category,completed,pending,refunded
0,clothing,44837.93,52348.46,51061.04
1,electronics,47317.56,46619.12,51790.64
2,food,58699.21,50951.93,61549.28
3,health,62758.34,53881.74,58400.02
4,travel,53794.60,58454.76,60497.66


In [16]:
# LIST aggregation — collect values into an array
# "what categories does each user buy from?"
con.sql("""
    SELECT user_id,
           LIST(DISTINCT category ORDER BY category) AS categories,
           COUNT(DISTINCT category)                  AS category_count
    FROM txns
    WHERE status = 'completed'
    GROUP BY user_id
    ORDER BY category_count DESC
    LIMIT 5
""").df()

,user_id,categories,category_count
0,74,"[clothing, electronics, food, health, travel]",5
1,91,"[clothing, electronics, food, health, travel]",5
2,67,"[clothing, electronics, food, health, travel]",5
3,16,"[clothing, electronics, food, health, travel]",5
4,26,"[clothing, electronics, food, health, travel]",5


In [17]:
# Window functions — running total and 7-day moving average
con.sql("""
    WITH daily AS (
        SELECT DATE_TRUNC('day', created_at)::DATE AS day,
               ROUND(SUM(amount), 2)               AS daily_revenue
        FROM txns
        WHERE status = 'completed'
        GROUP BY 1
    )
    SELECT day,
           daily_revenue,
           ROUND(SUM(daily_revenue)  OVER (ORDER BY day), 2)                          AS running_total,
           ROUND(AVG(daily_revenue)  OVER (ORDER BY day ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 2) AS ma_7d
    FROM daily
    ORDER BY day DESC
    LIMIT 14
""").df()

,day,daily_revenue,running_total,ma_7d
0,2026-06-17,2179.32,267407.64,2647.02
1,2026-06-16,4232.21,265228.32,2775.69
2,2026-06-15,1955.58,260996.11,2423.63
3,2026-06-14,2892.79,259040.53,2337.67
4,2026-06-13,2665.93,256147.74,2047.31
5,2026-06-12,2497.12,253481.81,2031.98
6,2026-06-11,2106.20,250984.69,2025.88
7,2026-06-10,3079.99,248878.49,2185.39
8,2026-06-09,1767.78,245798.50,2137.15
9,2026-06-08,1353.85,244030.72,2278.71


---
## 4 · PostgreSQL — attach and query directly

DuckDB's `postgres` extension attaches to the running Postgres container.  
You get full read access (and optionally write) over a standard SQL interface.

In [18]:
pg_con = duckdb.connect()

pg_con.execute("INSTALL postgres; LOAD postgres;")

pg_con.execute("""
    ATTACH 'dbname=sandbox user=spark password=spark123 host=postgres port=5432'
    AS pg (TYPE POSTGRES, READ_ONLY)
""")

# List tables in the attached database
pg_con.sql("SHOW TABLES FROM pg").show()

┌───────────────┐
│     name      │
│    varchar    │
├───────────────┤
│ daily_summary │
│ transactions  │
│ users         │
└───────────────┘



In [19]:
# Query Postgres tables with DuckDB SQL — no psycopg2, no ORM
pg_con.sql("""
    SELECT u.country,
           COUNT(DISTINCT t.id)   AS tx_count,
           ROUND(SUM(t.amount), 2) AS revenue
    FROM pg.transactions t
    JOIN pg.users u ON t.user_id = u.id
    WHERE t.status = 'completed'
    GROUP BY ALL
    ORDER BY revenue DESC
""").df()

,country,tx_count,revenue
0,SG,6,1110.00
1,MX,3,600.00
2,US,9,582.75
3,UK,6,305.97


In [20]:
# Cross-source query: join a Postgres table with a local pandas DataFrame
# Register the in-memory DataFrame into this connection first
pg_con.register('local_txns', txns_df)

pg_con.sql("""
    SELECT pg.users.country,
           ROUND(SUM(local_txns.amount), 2) AS generated_revenue
    FROM local_txns
    JOIN pg.users ON local_txns.user_id = pg.users.id
    GROUP BY ALL
    ORDER BY generated_revenue DESC
""").df()

,country,generated_revenue
0,DE,9559.33
1,US,9506.73
2,SG,8573.09
3,MX,7866.34
4,UK,5245.86


In [21]:
# Export a Postgres query result straight to Parquet
pg_con.sql("""
    COPY (
        SELECT * FROM pg.transactions
    ) TO '/home/jupyter/data/duck/pg_transactions.parquet' (FORMAT PARQUET)
""")

print('Exported Postgres table to Parquet.')

Exported Postgres table to Parquet.


---
## 5 · Persistent file

A `.duckdb` file stored in the mounted `./data/` directory survives container restarts.  
Treat it as a local data warehouse — write Spark results or Postgres snapshots into it.

In [22]:
db_path = '/home/jupyter/data/duck/warehouse.duckdb'

wh = duckdb.connect(db_path)

# Materialise a table from the Parquet file
wh.execute("""
    CREATE OR REPLACE TABLE transactions AS
    SELECT * FROM read_parquet('/home/jupyter/data/duck/txns.parquet')
""")

wh.execute("""
    CREATE OR REPLACE TABLE users AS
    SELECT * FROM read_csv('/home/jupyter/data/duck/users.csv')
""")

# Create a summary view
wh.execute("""
    CREATE OR REPLACE VIEW daily_revenue AS
    SELECT DATE_TRUNC('day', created_at)::DATE AS day,
           category,
           ROUND(SUM(amount), 2) AS revenue,
           COUNT(*)              AS tx_count
    FROM transactions
    WHERE status = 'completed'
    GROUP BY ALL
""") 

wh.sql("SHOW TABLES").show()
wh.close()
print(f'Saved to {db_path}')

┌───────────────┐
│     name      │
│    varchar    │
├───────────────┤
│ daily_revenue │
│ transactions  │
│ users         │
└───────────────┘

Saved to /home/jupyter/data/duck/warehouse.duckdb


In [23]:
# Reopen — tables and views are still there
wh = duckdb.connect(db_path, read_only=True)

wh.sql("""
    SELECT * FROM daily_revenue
    ORDER BY day DESC, revenue DESC
    LIMIT 10
""").df()

,day,category,revenue,tx_count
0,2026-06-17,clothing,1038.19,2
1,2026-06-17,food,783.52,1
2,2026-06-17,health,357.61,2
3,2026-06-16,health,1900.05,3
4,2026-06-16,clothing,1281.61,2
5,2026-06-16,travel,612.65,1
6,2026-06-16,food,437.90,1
7,2026-06-15,travel,744.20,2
8,2026-06-15,clothing,522.95,1
9,2026-06-15,health,378.89,1


---
## 6 · DuckDB ↔ pandas — round-trip patterns

The typical pattern: use **pandas** for row-level wrangling, **DuckDB** for SQL aggregations.

In [24]:
# pandas → DuckDB SQL → pandas
heavy_query = duckdb.sql("""
    WITH ranked AS (
        SELECT *,
               PERCENT_RANK() OVER (PARTITION BY category ORDER BY amount) AS pct_rank
        FROM txns_df
        WHERE status = 'completed'
    )
    SELECT category,
           ROUND(QUANTILE_CONT(amount, 0.50), 2) AS p50,
           ROUND(QUANTILE_CONT(amount, 0.90), 2) AS p90,
           ROUND(QUANTILE_CONT(amount, 0.99), 2) AS p99,
           COUNT(*) AS n
    FROM ranked
    GROUP BY ALL
    ORDER BY p90 DESC
""").df()   # ← back to pandas for plotting / further wrangling

heavy_query

,category,p50,p90,p99,n
0,electronics,354.52,743.84,786.88,120
1,travel,444.72,722.95,782.88,128
2,clothing,399.90,716.43,785.12,111
3,health,451.90,710.00,782.98,149
4,food,395.16,709.25,781.54,142


In [25]:
# Relation API — build queries programmatically (like a query builder)
rel = (
    duckdb.from_df(txns_df)
          .filter("status = 'completed'")
          .aggregate("category, ROUND(SUM(amount),2) AS revenue, COUNT(*) AS n", "category")
          .order("revenue DESC")
)

# Inspect the SQL it generated
print(rel.query('txns_df', 'SELECT * FROM txns_df'))
rel.df()

┌─────────────┬──────────┬───────┐
│  category   │ revenue  │   n   │
│   varchar   │  double  │ int64 │
├─────────────┼──────────┼───────┤
│ health      │ 62758.34 │   149 │
│ food        │ 58699.21 │   142 │
│ travel      │  53794.6 │   128 │
│ electronics │ 47317.56 │   120 │
│ clothing    │ 44837.93 │   111 │
└─────────────┴──────────┴───────┘



,category,revenue,n
0,health,62758.34,149
1,food,58699.21,142
2,travel,53794.60,128
3,electronics,47317.56,120
4,clothing,44837.93,111


In [26]:
# Export query results to multiple formats
result_df = duckdb.sql("SELECT * FROM txns_df WHERE status = 'completed'").df()

# Parquet
duckdb.sql("COPY (SELECT * FROM txns_df) TO '/home/jupyter/data/duck/export.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)")

# CSV
duckdb.sql("COPY (SELECT * FROM txns_df) TO '/home/jupyter/data/duck/export.csv' (HEADER, DELIMITER ',')")

# JSON
duckdb.sql("COPY (SELECT * FROM txns_df LIMIT 10) TO '/home/jupyter/data/duck/export.json' (FORMAT JSON, ARRAY true)")

print('Exports written to /home/jupyter/data/duck/')

BinderException: Binder Error: Referenced column "status" not found in FROM clause!
Candidate bindings: "revenue"

---
## Quick-reference cheatsheet

```python
import duckdb

# ── Connections ──────────────────────────────────────────────────
duckdb.sql("...")                        # default in-memory, one-shot
con = duckdb.connect()                   # reusable in-memory connection
con = duckdb.connect('my.duckdb')        # persistent file
con = duckdb.connect('my.duckdb', read_only=True)

# ── Return formats ───────────────────────────────────────────────
con.sql("...").df()                      # pandas DataFrame
con.sql("...").pl()                      # polars DataFrame
con.sql("...").arrow()                   # PyArrow Table
con.sql("...").fetchall()                # list of tuples
con.sql("...").show()                    # print to console

# ── Reading ──────────────────────────────────────────────────────
read_csv('file.csv')
read_csv('*.csv', union_by_name=True)    # multiple files, auto-union
read_parquet('dir/**/*.parquet')         # recursive glob
read_json('events.jsonl')

# ── Writing ──────────────────────────────────────────────────────
COPY (...) TO 'out.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
COPY (...) TO 'out.csv'     (HEADER, DELIMITER ',')

# ── Python objects ───────────────────────────────────────────────
con.register('name', df)                 # register pandas/arrow as view
duckdb.from_df(df).sql("...")           # relation from DataFrame
duckdb.arrow(arrow_table)               # relation from Arrow

# ── Attach Postgres ──────────────────────────────────────────────
con.execute("INSTALL postgres; LOAD postgres;")
con.execute("ATTACH 'dbname=... host=...' AS pg (TYPE POSTGRES)")
con.sql("SELECT * FROM pg.my_table")

# ── Handy SQL extensions ─────────────────────────────────────────
SUMMARIZE table_name                     -- column stats
GROUP BY ALL                             -- group by non-aggregated cols
SELECT * EXCLUDE (col1, col2)
SELECT * REPLACE (expr AS col)
QUALIFY <window_condition>               -- filter on window result
PIVOT ... ON col USING agg GROUP BY ...
LIST(col ORDER BY ...)                   -- array aggregation
QUANTILE_CONT(col, 0.95)                 -- exact percentile
```